# Amazon ML Challenge 2026 — Notebook 3: Self-Training, Graph Consistency & Submission
### France Domain Adaptation, Maximum-Weight Bipartite Cardinality, and Leaderboard Generation

This notebook executes the final inference stages:
1. **Test Feature Extraction:** Rapid vectorization of test candidate pairs.
2. **3-Stage Cascade Scoring:** High-confidence rule filter + LightGBM probability inference.
3. **Novel Self-Training Domain Adaptation for France:** Iterative pseudo-label bootstrapping ($	au_{pos}=0.92, 	au_{neg}=0.08$) to close the zero-shot country gap.
4. **Graph-Based Post-Processing:**
   - Cardinality resolution via greedy Maximum Weighted Bipartite Matching.
   - Singleton verification ($	au_{singleton}=0.30$).
   - Match count capping ($\le 11$).
5. **Submission Generation & Local Validation:** Produces `matching_results.tsv` and runs `validate_submission.py`.


In [ ]:
# Setup and Imports
!pip install -q rapidfuzz Metaphone lightgbm polars
import os
import gc
import re
import time
import pickle
import numpy as np
import pandas as pd
import lightgbm as lgb
from collections import defaultdict
from rapidfuzz import fuzz, distance

print("Ready for Test Inference & Submission Generation!")


## 1. Load Data, Candidates & Trained Model


In [ ]:
if os.path.exists('/kaggle/input/ml-challenge-2026-dataset'):
    DATA_DIR = '/kaggle/input/ml-challenge-2026-dataset'
elif os.path.exists('D:/ML_Challenge/DATA/UNZIPPED/student_resource/dataset'):
    DATA_DIR = 'D:/ML_Challenge/DATA/UNZIPPED/student_resource/dataset'
else:
    DATA_DIR = './dataset'

CANDIDATES_PATH = './output/candidate_pairs.tsv' if os.path.exists('./output/candidate_pairs.tsv') else '/kaggle/input/ml-challenge-candidate-pairs/candidate_pairs.tsv'
MODEL_PATH = './lgbm_entity_resolution.pkl' if os.path.exists('./lgbm_entity_resolution.pkl') else '/kaggle/input/ml-challenge-trained-model/lgbm_entity_resolution.pkl'

print(f"Data Dir: {DATA_DIR}")
print(f"Candidates: {CANDIDATES_PATH}")
print(f"Model: {MODEL_PATH}")


## 2. Self-Training Domain Adaptation Engine for France


In [ ]:
def run_france_self_training(lgbm_model, france_X, france_pairs, max_iters=2, tau_pos=0.92, tau_neg=0.08):
    print(f"\n--- Running France Self-Training Adaptation ({len(france_pairs):,} candidate pairs) ---")
    
    current_model = lgbm_model
    for it in range(1, max_iters + 1):
        probs = current_model.predict(france_X)
        
        pos_mask = (probs >= tau_pos)
        neg_mask = (probs <= tau_neg)
        
        n_pos = np.sum(pos_mask)
        n_neg = np.sum(neg_mask)
        print(f"Iteration {it}: Found {n_pos:,} high-confidence pseudo-positives and {n_neg:,} pseudo-negatives.")
        
        if n_pos < 1000:
            print("Insufficient high-confidence pseudo-labels. Stopping self-training.")
            break
            
        # Subsample negatives to maintain 1:5 ratio
        neg_indices = np.where(neg_mask)[0]
        sample_neg_indices = np.random.choice(neg_indices, size=min(len(neg_indices), n_pos * 5), replace=False)
        
        pseudo_X = np.vstack([france_X[pos_mask], france_X[sample_neg_indices]])
        pseudo_y = np.hstack([np.ones(n_pos, dtype=np.float32), np.zeros(len(sample_neg_indices), dtype=np.float32)])
        
        # Sample weights: 0.5 for pseudo-labels to avoid confirmation bias
        weights = np.full(len(pseudo_y), 0.5, dtype=np.float32)
        
        train_data = lgb.Dataset(pseudo_X, label=pseudo_y, weight=weights)
        current_model = lgb.train(
            {'objective': 'binary', 'learning_rate': 0.03, 'num_leaves': 63, 'max_depth': 7, 'verbose': -1},
            train_data,
            num_boost_round=400,
            init_model=current_model
        )
        print(f"Iteration {it} complete.")
        
    return current_model


## 3. Graph Consistency & Cardinality Post-Processing


In [ ]:
def post_process_predictions(pair_predictions, tau_match=0.65, tau_singleton=0.30, max_matches=11):
    print("\nRunning Graph-Based Consistency Post-Processing...")
    
    # 1. Sort all predictions descending by probability
    pair_predictions.sort(key=lambda x: x[2], reverse=True)
    
    # 2. Greedy Maximum Weighted Bipartite Matching (Cardinality enforcement: S2/S3 entity maps to at most 1 S1)
    assigned_cand = set()
    s1_matches = defaultdict(set)
    s1_max_prob = defaultdict(float)
    
    for s1_id, cand_id, prob in pair_predictions:
        s1_max_prob[s1_id] = max(s1_max_prob[s1_id], prob)
        if prob >= tau_match and cand_id not in assigned_cand:
            s1_matches[s1_id].add(cand_id)
            assigned_cand.add(cand_id)
            
    # 3. Singleton Protection
    for s1_id, max_p in s1_max_prob.items():
        if max_p < tau_singleton:
            s1_matches[s1_id] = set()
            
    # 4. Cap Maximum Matches to 11
    for s1_id in list(s1_matches.keys()):
        if len(s1_matches[s1_id]) > max_matches:
            # retain top matches
            sorted_m = sorted(s1_matches[s1_id], key=lambda cid: next(p for s, c, p in pair_predictions if s==s1_id and c==cid), reverse=True)
            s1_matches[s1_id] = set(sorted_m[:max_matches])
            
    print("Post-processing complete!")
    return s1_matches


## 4. Export `matching_results.tsv` and Run Submission Validator


In [ ]:
def generate_final_submission(s1_test_path, final_matches_dict, output_path):
    s1_df = pd.read_csv(s1_test_path, sep='\t')
    print(f"Writing {len(s1_df):,} rows to {output_path}...")
    
    singletons = 0
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write("source1_entity_id\tmatched_entity_ids\n")
        for sid in s1_df['entity_id']:
            matches = final_matches_dict.get(sid, set())
            if not matches:
                singletons += 1
                f.write(f"{sid}\t\n")
            else:
                f.write(f"{sid}\t{','.join(sorted(matches))}\n")
                
    print(f"Generated matching_results.tsv! Singletons: {singletons:,} ({singletons/len(s1_df)*100:.2f}%)")
    
    # Run official validator
    validator_path = os.path.join(os.path.dirname(s1_test_path), '../../utils/validate_submission.py')
    if not os.path.exists(validator_path):
        validator_path = './utils/validate_submission.py'
        
    if os.path.exists(validator_path):
        print("\nExecuting official validator script:")
        os.system(f"python {validator_path} --matching {output_path} --test-dir {os.path.dirname(s1_test_path)}")
